In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os
import json

In [ ]:
# import getpass
LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING", "false").lower() == "true"
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY", None)

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "kimi-k2:1t-cloud")
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY", "ollama")
OLLAMA_MODEL_URL = os.getenv("OLLAMA_MODEL_URL", "http://localhost:11434")

print(f"LANGSMITH_TRACING: {LANGSMITH_TRACING}")
print(f"LANGSMITH_API_KEY: {LANGSMITH_API_KEY or 'not set'}")
print(f"OLLAMA_MODEL: {OLLAMA_MODEL}")
print(f"OLLAMA_API_KEY: {OLLAMA_API_KEY or 'not set'}")
print(f"OLLAMA_MODEL_URL: {OLLAMA_MODEL_URL}")

args = {
  "model": OLLAMA_MODEL,
  "model_provider": "openai",
  "api_key": OLLAMA_API_KEY,
  "base_url": f"{OLLAMA_MODEL_URL}/v1",
}

model = init_chat_model(
  args["model"],
  model_provider=args["model_provider"],
  api_key=args["api_key"],
  base_url=args["base_url"],
)

print(json.dumps(args, indent=2))

# 1. Define custom state
First, define a custom state schema that tracks which step is currently active:

The current_step field is the core of the state machine pattern - it determines which configuration (prompt + tools) is loaded on each turn.

In [ ]:
from langchain.agents import AgentState
from typing_extensions import NotRequired
from typing import Literal

WorkflowStep = Literal["receptionist", "sports_data_collector", "content_editor", "betting_editor"]
RequestType = Literal["games", "expert_picks"]

class SupportState(AgentState):
    """State for sports games and betting picks workflow."""
    current_step: NotRequired[WorkflowStep]
    request_type: NotRequired[RequestType]
    picks_status: NotRequired[Literal["UPCOMING", "PAST"]]
    league: NotRequired[str]
    start_date: NotRequired[str]
    end_date: NotRequired[str]
    games: NotRequired[list[dict]]

# 2. Create tools that manage workflow state

Create tools that update the workflow state. The receptionist now routes to one of two intents:
- `games` → sports data collector flow
- `expert_picks` → betting editor flow

The key is using `Command` to update state, including both `request_type` and `current_step` for clean branching.

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command
import json
import requests

EXPERT_PICKS_QUERY = """query SLUI_ExpertPicks($first: Int = 10, $after: String, $sortBy: [ExpertPickSortByInput!], $filter: ExpertPickFilterInput) {
  expertPicks(first: $first, after: $after, sortBy: $sortBy, filter: $filter) {
    totalCount
    pageInfo {
      startCursor
      endCursor
      hasNextPage
    }
    edges {
      cursor
      node {
        id
        isFeatured
        locked
        createdAt
        resultStatus
        unit
        writeup
        sportsbookName
        expert {
            id
            firstName
            lastName
        }
      }
    }
  }
}
"""

@tool
def request_expert_picks(
    status: Literal["UPCOMING", "PAST"],
    league: str
    ) -> str:
    """Request expert picks for a given sports league. The league parameter can be 'NFL', 'NBA', 'MLB'."""
    url = "https://helios.cbssports.com/"
    headers = {
        "accept": "*/*",
        "accept-language": "en-US,en;q=0.9,fr;q=0.8,es;q=0.7,zh-CN;q=0.6,zh;q=0.5,pt;q=0.4,de;q=0.3,la;q=0.2",
        "cache-control": "no-cache",
        "content-type": "application/json",
        "helios-client-name": "sportsline-web",
        "origin": "https://www.sportsline.com",
        "pid": "L:1:wNjylykbeW%252BQ7HXfD0npOcRWsoZTVKd9G9Ivc3FOgpukRr0i4BBoWiN8DdgiuIbo:1",
        "pragma": "no-cache",
        "priority": "u=1, i",
        "referer": "https://www.sportsline.com/",
        "sec-ch-ua": '"Not:A-Brand";v="99", "Google Chrome";v="145", "Chromium";v="145"',
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": '"macOS"',
        "sec-fetch-dest": "empty",
        "sec-fetch-mode": "cors",
        "sec-fetch-site": "cross-site",
        "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
    }
    payload = {
        "operationName": "SLUI_ExpertPicks",
        "variables": {
            "sortBy": [
                {"order": "DESC", "field": "FEATURED"},
                {"order": "DESC", "field": "CREATED"},
            ],
            "filter": {"state": status, "leagues": [league]},
        },
        "query": EXPERT_PICKS_QUERY,
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=15)
        response.raise_for_status()
        data = response.json()
        edges = (
            data.get("data", {})
            .get("expertPicks", {})
            .get("edges", [])
        )

        league_upper = (league or "").upper()
        status_upper = (status or "UPCOMING").upper()

        picks = []
        for edge in edges:
            node = edge.get("node", {})
            expert = node.get("expert", {})
            first_name = expert.get("firstName", "").strip()
            last_name = expert.get("lastName", "").strip()
            full_name = f"{first_name} {last_name}".strip() or "n/a"

            picks.append(
                {
                    "expert_name": full_name,
                    "writeup": node.get("writeup", "n/a"),
                    "unit": node.get("unit", None),
                    "sportsbook": node.get("sportsbookName", "n/a"),
                    "result_status": node.get("resultStatus", "n/a"),
                    "created_at": node.get("createdAt", "n/a"),
                }
            )

        result = {
            "league": league_upper,
            "status": status_upper,
            "total_picks": data.get("data", {}).get("expertPicks", {}).get("totalCount", 0),
            "returned_edges": len(picks),
            "picks": picks,
        }
        return json.dumps(result, indent=2)
    except requests.HTTPError:
        return f"HTTP {response.status_code}: {response.text[:2000]}"
    except requests.RequestException as e:
        return f"Error fetching expert picks for {league}: {e}"

In [ ]:
# primpy url
PRIMPY_TOKEN = os.getenv("PRIMPY_TOKEN", "")
PRIMPY_URL = "https://api.cbssports.com/primpy/sportsline"

@tool
def capture_user_request(
    request_type: Literal["games", "expert_picks"],
    league: str,
    start_date: str = "",
    end_date: str = "",
    picks_status: Literal["UPCOMING", "PAST"] = "UPCOMING",
    runtime: ToolRuntime[SupportState] = None,
) -> Command:
    """Capture user intent and route to either games flow or expert picks (betting) flow.

    - request_type='games' requires start_date and end_date in YYYYMMDD format.
    - request_type='expert_picks' routes to betting editor and uses picks_status.
    """
    normalized_type = (request_type or "").lower()
    league_upper = (league or "").upper()
    picks_status = (picks_status or "UPCOMING").upper()

    if normalized_type == "games" and (not start_date or not end_date):
        tool_message = ToolMessage(
            content="For games requests, please provide both start_date and end_date in YYYYMMDD.",
            tool_call_id=runtime.tool_call_id,
        )
        return Command(
            update={
                "messages": [tool_message],
                "current_step": "receptionist",
            }
        )

    next_step = "sports_data_collector" if normalized_type == "games" else "betting_editor"
    tool_message = ToolMessage(
        content=(
            f"Captured request_type={normalized_type}, league={league_upper}, "
            f"start_date={start_date or 'n/a'}, end_date={end_date or 'n/a'}, picks_status={picks_status}."
        ),
        tool_call_id=runtime.tool_call_id,
    )

    return Command(
        update={
            "request_type": normalized_type,
            "league": league_upper,
            "start_date": start_date or None,
            "end_date": end_date or None,
            "picks_status": picks_status,
            "current_step": next_step,
            "messages": [tool_message],
        }
    )

@tool
def request_games_by_league(
    league: str,
    start_date: str,
    end_date: str,
    runtime: ToolRuntime[SupportState],
) -> Command:
    """Fetch upcoming games by league and date range, then hand off to content editor."""
    if not PRIMPY_TOKEN:
        tool_message = ToolMessage(
            content="Missing PRIMPY_TOKEN. Set it in your environment before requesting games.",
            tool_call_id=runtime.tool_call_id,
        )
        return Command(update={"messages": [tool_message]})

    try:
        league_code = league.lower()
        url = (
            f"{PRIMPY_URL}/league/games/{league_code}"
            f"?resources=weather,gameOdds,venue,homeTeam,awayTeam"
            f"&startDate={start_date}&endDate={end_date}&access_token={PRIMPY_TOKEN}"
        )
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        data = response.json()

        games = [
            {
                "home_team": game.get("homeTeam", {}).get("mediumName", "n/a"),
                "away_team": game.get("awayTeam", {}).get("mediumName", "n/a"),
                "start_time": game.get("scheduledTime", "n/a"),
            }
            for game in data.get("data", [])
        ]

        tool_message = ToolMessage(
            content=json.dumps(
                {
                    "league": league.upper(),
                    "start_date": start_date,
                    "end_date": end_date,
                    "games_found": len(games),
                }
            ),
            tool_call_id=runtime.tool_call_id,
        )

        return Command(
            update={
                "league": league.upper(),
                "start_date": start_date,
                "end_date": end_date,
                "games": games,
                "current_step": "content_editor",
                "messages": [tool_message],
            }
        )
    except requests.HTTPError:
        status_code = response.status_code if "response" in locals() else "unknown"
        error_text = response.text[:2000] if "response" in locals() else "no response body"
        tool_message = ToolMessage(
            content=f"HTTP {status_code}: {error_text}",
            tool_call_id=runtime.tool_call_id,
        )
        return Command(update={"messages": [tool_message]})
    except requests.RequestException as error:
        tool_message = ToolMessage(
            content=f"Error fetching games for {league.upper()}: {error}",
            tool_call_id=runtime.tool_call_id,
        )
        return Command(update={"messages": [tool_message]})

# 3. Define step configurations

Define prompts and tools for each step. First, define the prompts for each step:

In [ ]:
# Define prompts as constants for easy reference
RECEPTIONIST_PROMPT = """You are the receptionist for a sports assistant workflow.

CURRENT STAGE: Receptionist

Your job in this step:
1. Identify intent: either `games` or `expert_picks`.
2. Always collect `league`.
3. If intent is `games`, also collect `start_date` and `end_date` in YYYYMMDD format.
4. If intent is `expert_picks`, collect optional `picks_status` (default UPCOMING).
5. Once required fields are ready, call `capture_user_request` to store and route.

Be friendly, concise, and ask only for missing fields."""

SPORTS_DATA_COLLECTOR_PROMPT = """You are the sports data collector.

CURRENT STAGE: Sports Data Collector

Only handle requests where request_type is `games`.
Use `request_games_by_league` with league, start_date, and end_date from state.
Do not ask the user more questions in this step. Fetch data and hand off to content editor."""

BETTING_EDITOR_PROMPT = """You are the betting editor for expert picks.

CURRENT STAGE: Betting Editor

Only handle requests where request_type is `expert_picks`.
Always call `request_expert_picks` first with:
- status = picks_status (default UPCOMING)
- league = league

Then provide a concise betting-focused summary that includes:
- league and pick status
- total picks available
- a bullet list of expert names from `picks[].expert_name`
- top insights from the writeups (cite expert names)
- a brief responsible-betting note

If `picks` is empty, clearly say no picks are currently available."""

CONTENT_EDITOR_PROMPT = """You are the content editor.

CURRENT STAGE: Content Editor

Only handle requests where request_type is `games`.
Write a friendly final response that includes:
- league
- date range (start_date to end_date)
- a short summary of all upcoming games in `games`
- if no games were found, say so clearly and politely
"""

Then map step names to their configurations using a dictionary:

In [ ]:
# Step configuration: maps step name to (prompt, tools, required_state)
STEP_CONFIG = {
    "receptionist": {
        "prompt": RECEPTIONIST_PROMPT,
        "tools": [capture_user_request],
        "requires": [],
        "icon": "🛎️",
    },
    "sports_data_collector": {
        "prompt": SPORTS_DATA_COLLECTOR_PROMPT,
        "tools": [request_games_by_league],
        "requires": ["request_type", "league", "start_date", "end_date"],
        "icon": "📊",
    },
    "betting_editor": {
        "prompt": BETTING_EDITOR_PROMPT,
        "tools": [request_expert_picks],
        "requires": ["request_type", "league"],
        "icon": "💸",
    },
    "content_editor": {
        "prompt": CONTENT_EDITOR_PROMPT,
        "tools": [],
        "requires": ["request_type", "league", "start_date", "end_date", "games"],
        "icon": "✍️",
    },
}

This dictionary-based configuration makes it easy to:
* See all steps at a glance
* Add new intent-specific steps (like `betting_editor`)
* Understand workflow dependencies (`requires` field)
* Route safely by request type (`games` vs `expert_picks`)

# 4. Create step-based middleware

Create middleware that reads `current_step` from state and applies the appropriate configuration. We’ll use the `@wrap_model_call` decorator for a clean implementation:

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def apply_step_config(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """Configure agent behavior based on the current step."""
    current_step = request.state.get("current_step", "receptionist")
    stage_config = STEP_CONFIG[current_step]

    for key in stage_config["requires"]:
        if request.state.get(key) is None:
            raise ValueError(f"{key} must be set before reaching {current_step}")

    request_type = request.state.get("request_type")
    if current_step in {"sports_data_collector", "content_editor"} and request_type != "games":
        raise ValueError(f"{current_step} only supports request_type='games'")
    if current_step == "betting_editor" and request_type != "expert_picks":
        raise ValueError("betting_editor only supports request_type='expert_picks'")

    system_prompt = stage_config["prompt"].format(**request.state)

    request = request.override(
        system_prompt=system_prompt,
        tools=stage_config["tools"],
    )

    return handler(request)

This middleware:
1. Reads current step: Gets `current_step` from state (defaults to `receptionist`).
2. Looks up configuration: Finds the matching entry in `STEP_CONFIG`.
3. Validates dependencies: Ensures required state fields exist.
4. Formats prompt: Injects state values into the prompt template.
5. Applies configuration: Overrides the system prompt and available tools.

The `request.override()` method is key - it allows us to dynamically change the agent’s behavior based on state without creating separate agent instances.

# 5. Create the agent

Now create the agent with the step-based middleware and a checkpointer for state persistence:

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Collect all tools from all step configurations
all_tools = [
    capture_user_request,
    request_games_by_league,
    request_expert_picks,
]

# Create the agent with step-based configuration
agent = create_agent(
    model,
    tools=all_tools,
    state_schema=SupportState,
    middleware=[apply_step_config],
    checkpointer=InMemorySaver(),
)

In [ ]:
from IPython.display import Image, display

display(Image(agent.get_graph().draw_mermaid_png()))

# 6. Test the workflow

Test both supported intents:
1. Games flow (`games`): receptionist → sports data collector → content editor
2. Expert picks flow (`expert_picks`): receptionist → betting editor

In [ ]:
from langchain.messages import HumanMessage
import uuid

# Start a new conversation thread
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

print("👋 Welcome to the Sports Assistant test run!")
print("This workflow supports 2 user intents:")
print("  • games -> receptionist → sports_data_collector → content_editor")
print("  • expert_picks -> receptionist → betting_editor")
print("-" * 70)

def run_turn(user_text: str):
    print(f"\n🧑 User: {user_text}")
    result = agent.invoke(
        {"messages": [HumanMessage(user_text)]},
        config,
    )

    print("🔎 Tool transitions this turn:")
    tool_seen = False
    for message in result.get("messages", []):
        if message.type == "tool":
            tool_seen = True
            print(f"  • {message.content}")
    if not tool_seen:
        print("  • No tool call this turn")

    current_step = result.get("current_step", "receptionist")
    icon = STEP_CONFIG.get(current_step, {}).get("icon", "")
    print(f"📍 Current step: {icon} {current_step}")
    print(f"🧭 Request type: {result.get('request_type', 'not set')}")

    print("🤖 Assistant:")
    for message in result.get("messages", [])[::-1]:
        if message.type == "ai" and message.content:
            print(message.content)
            break

    games = result.get("games", [])
    print(f"📅 Games in state: {len(games)}")
    return result

# -------- Test 1: Games flow --------
league = "NBA"
start_date = "20260307"
end_date = "20260308"
games_input = f"I want games for league {league} from {start_date} to {end_date}."
print(f"Games test input -> {games_input}")
print("-" * 70)

result = run_turn(games_input)

if result.get("current_step") == "sports_data_collector":
    result = run_turn(
        f"Please call request_games_by_league with league={league}, start_date={start_date}, end_date={end_date}."
    )

if result.get("current_step") == "content_editor":
    result = run_turn("Please provide the final friendly summary of upcoming games.")

# -------- Test 2: Expert picks / betting flow --------
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}
picks_input = "I want upcoming expert picks for NBA."
print("\n" + "=" * 70)
print(f"Expert picks test input -> {picks_input}")
print("-" * 70)

result = run_turn(picks_input)

if result.get("current_step") == "betting_editor":
    league_for_picks = result.get("league", "NBA")
    picks_status = result.get("picks_status", "UPCOMING")
    result = run_turn(
        f"Please call request_expert_picks with status={picks_status} and league={league_for_picks}, then provide a betting-focused summary."
    )

print("\n✅ Test run complete.")

In [ ]:
# Focused expert-picks test (with expert-name requirement)
from langchain.messages import HumanMessage
import uuid

thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

def run_expert_turn(user_text: str):
    print(f"\n🧑 User: {user_text}")
    result = agent.invoke(
        {"messages": [HumanMessage(user_text)]},
        config,
    )
    for message in result.get("messages", []):
        if message.type == "tool":
            print(f"🔧 Tool: {message.content}")

    print(f"📍 Current step: {result.get('current_step', 'receptionist')}")
    print("🤖 Assistant:")
    for message in result.get("messages", [])[::-1]:
        if message.type == "ai" and message.content:
            print(message.content)
            break
    return result

result = run_expert_turn("I want upcoming expert picks for NBA.")
if result.get("current_step") == "betting_editor":
    league_for_picks = result.get("league", "NBA")
    picks_status = result.get("picks_status", "UPCOMING")
    run_expert_turn(
        f"Call request_expert_picks with status={picks_status} and league={league_for_picks}, then summarize and include expert names from the picks."
    )